# Smart Farming Assistant

# Sistem Rekomendasi Tanaman Menggunakan Random Forest Classifier

---

## Nama Project

**Smart Farming Assistant**

---

## Disusun Oleh

**Nama : Junaidy Muhtarom**

**NIM : 2305101101**

**Program Studi : Teknik Informatika**

---

## Deskripsi

Notebook ini merupakan proses pembangunan model Machine Learning pada aplikasi **Smart Farming Assistant**.

Model yang digunakan adalah **Random Forest Classifier** untuk memberikan rekomendasi tanaman berdasarkan kondisi tanah dan lingkungan.

Model akan dilatih menggunakan **Crop Recommendation Dataset** yang berisi 2.200 data dan 22 kelas tanaman.

Output dari notebook ini berupa dua file yaitu:

- crop_model.pkl
- label_encoder.pkl

yang selanjutnya digunakan pada aplikasi berbasis **Streamlit**.

# 1. Import Library

Pada tahap ini dilakukan proses import seluruh library yang dibutuhkan selama pembangunan model Machine Learning.

Library yang digunakan meliputi:

- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-Learn
- Joblib

Library tersebut digunakan untuk proses membaca dataset, visualisasi data, preprocessing, pelatihan model, evaluasi model, hingga penyimpanan model.

In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

import joblib

import warnings
warnings.filterwarnings("ignore")

# 2. Load Dataset

Tahapan berikutnya adalah membaca dataset yang digunakan dalam penelitian.

Dataset yang digunakan merupakan **Crop Recommendation Dataset** yang diperoleh dari Kaggle.

Dataset memiliki tujuh parameter utama yang menjadi input model, yaitu:

- Nitrogen (N)
- Phosphorus (P)
- Potassium (K)
- Temperature
- Humidity
- pH
- Rainfall

Sedangkan output dari dataset berupa **22 jenis tanaman**.

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv("Crop_recommendation.csv")

print("="*60)
print("DATASET BERHASIL DIMUAT")
print("="*60)

df.head()

In [ ]:
# ============================================================
# UKURAN DATASET
# ============================================================

print("Jumlah Baris :", df.shape[0])
print("Jumlah Kolom :", df.shape[1])

print()

print(df.columns)

# 3. Menampilkan Informasi Dataset

Tahapan ini bertujuan untuk mengetahui struktur dataset.

Informasi yang ditampilkan meliputi:

- Jumlah data
- Nama kolom
- Tipe data
- Missing Value

Informasi tersebut sangat penting sebelum melakukan proses preprocessing.

In [ ]:
# ============================================================
# INFORMASI DATASET
# ============================================================

print("="*60)
print("INFORMASI DATASET")
print("="*60)

print()

df.info()

In [ ]:
# ============================================================
# CEK MISSING VALUE
# ============================================================

print("="*60)
print("MISSING VALUE")
print("="*60)

df.isnull().sum()

# ============================================================
# DESKRIPSI DATA
# ============================================================

df.describe().T

# 4. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) merupakan tahapan untuk memahami karakteristik dataset sebelum dilakukan proses pelatihan model Machine Learning.

Pada tahap ini dilakukan beberapa analisis, antara lain:

- Mengetahui jumlah masing-masing jenis tanaman.
- Menampilkan distribusi data.
- Menganalisis hubungan antar fitur.
- Mengetahui karakteristik setiap parameter.

Hasil EDA digunakan sebagai dasar dalam menentukan proses preprocessing dan pelatihan model.

In [ ]:
# ============================================================
# MENAMPILKAN LIMA DATA PERTAMA
# ============================================================

df.head()

In [ ]:
# ============================================================
# MENAMPILKAN LIMA DATA TERAKHIR
# ============================================================

df.tail()

# 5. Distribusi Jumlah Tanaman

Tahapan ini bertujuan untuk mengetahui jumlah data pada setiap kelas tanaman.

Visualisasi dilakukan menggunakan diagram batang sehingga lebih mudah mengetahui apakah dataset memiliki distribusi yang seimbang.

In [ ]:
# ============================================================
# JUMLAH DATA SETIAP TANAMAN
# ============================================================

crop_count = df["label"].value_counts()

crop_count

In [ ]:
# ============================================================
# VISUALISASI JUMLAH TANAMAN
# ============================================================

plt.figure(figsize=(14,6))

crop_count.plot(
    kind="bar",
    color="green"
)

plt.title("Jumlah Data Setiap Tanaman")

plt.xlabel("Jenis Tanaman")

plt.ylabel("Jumlah Data")

plt.xticks(rotation=45)

plt.show()

# 6. Distribusi Seluruh Parameter

Visualisasi histogram digunakan untuk melihat penyebaran data pada setiap fitur.

Dengan histogram dapat diketahui apakah suatu fitur memiliki distribusi yang merata atau terdapat kecenderungan tertentu.

In [ ]:
# ============================================================
# HISTOGRAM
# ============================================================

df.hist(

figsize=(15,10),

bins=20

)

plt.tight_layout()

plt.show()

# 7. Korelasi Antar Fitur

Tahapan ini digunakan untuk mengetahui hubungan antar parameter yang terdapat pada dataset.

Analisis dilakukan menggunakan Heatmap Correlation.

Semakin tinggi nilai korelasi maka semakin kuat hubungan antar fitur.

In [ ]:
# ============================================================
# HEATMAP
# ============================================================

plt.figure(figsize=(10,8))

sns.heatmap(

df.drop("label",axis=1).corr(),

annot=True,

cmap="Greens"

)

plt.title("Correlation Matrix")

plt.show()

# 8. Kesimpulan Exploratory Data Analysis

Berdasarkan hasil Exploratory Data Analysis diperoleh beberapa informasi penting, yaitu:

- Dataset memiliki jumlah data sebanyak 2.200 data.
- Dataset terdiri dari 22 jenis tanaman.
- Tidak ditemukan missing value.
- Seluruh parameter memiliki tipe data numerik.
- Distribusi jumlah data antar kelas relatif seimbang sehingga dataset layak digunakan untuk proses pelatihan model Machine Learning.

# 9. Preprocessing Data

Sebelum model Machine Learning dilatih, dataset perlu melalui tahap preprocessing.

Pada penelitian ini preprocessing dilakukan dengan cara:

1. Memisahkan fitur (X) dan label (y).
2. Melakukan Label Encoding pada data target.
3. Menyiapkan data agar dapat digunakan pada proses pelatihan model.

Tahapan preprocessing sangat penting karena algoritma Random Forest hanya dapat memproses data numerik.

In [ ]:
# ============================================================
# MEMISAHKAN FITUR DAN LABEL
# ============================================================

X = df.drop("label", axis=1)
y = df["label"]

print("Jumlah Data Fitur :", X.shape)
print("Jumlah Data Label :", y.shape)

# 10. Label Encoding

Kolom **label** masih berupa data kategorikal (nama tanaman).

Agar dapat diproses oleh algoritma Random Forest, data tersebut diubah menjadi bentuk numerik menggunakan **LabelEncoder** dari Scikit-Learn.

Hasil encoding nantinya juga akan disimpan sehingga aplikasi Streamlit dapat mengubah kembali hasil prediksi menjadi nama tanaman.

In [ ]:
# ============================================================
# LABEL ENCODING
# ============================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Encoding berhasil dilakukan.")

In [ ]:
# ============================================================
# MENAMPILKAN HASIL ENCODING
# ============================================================

encoding_df = pd.DataFrame({
    "Jenis Tanaman": label_encoder.classes_,
    "Kode": range(len(label_encoder.classes_))
})

encoding_df

# 11. Membagi Data Training dan Testing

Dataset kemudian dibagi menjadi dua bagian yaitu:

- Data Training sebesar **80%**
- Data Testing sebesar **20%**

Data training digunakan untuk melatih model, sedangkan data testing digunakan untuk mengevaluasi kemampuan model dalam melakukan prediksi terhadap data baru.

In [ ]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.2,

    random_state=42,

    stratify=y_encoded

)

print("Data Training :", X_train.shape)
print("Data Testing  :", X_test.shape)

# 12. Membangun Model Random Forest

Random Forest merupakan algoritma Machine Learning berbasis ensemble yang terdiri dari banyak Decision Tree.

Keunggulan Random Forest antara lain:

- Memiliki akurasi tinggi.
- Mampu menangani data dalam jumlah besar.
- Tidak mudah mengalami overfitting.
- Stabil terhadap variasi data.

Pada penelitian ini digunakan sebanyak **200 Decision Tree** agar model memiliki performa yang lebih baik.

In [ ]:
# ============================================================
# MEMBANGUN MODEL RANDOM FOREST
# ============================================================

model = RandomForestClassifier(

    n_estimators=200,

    random_state=42

)

In [ ]:
# ============================================================
# MELATIH MODEL
# ============================================================

model.fit(

    X_train,

    y_train

)

print("Model Random Forest berhasil dilatih.")

# 13. Informasi Model

Setelah proses pelatihan selesai, model Random Forest siap digunakan untuk melakukan prediksi.

Tahapan berikutnya adalah melakukan evaluasi terhadap model menggunakan data testing untuk mengetahui tingkat akurasi dan performa model.

In [ ]:
# ============================================================
# INFORMASI MODEL
# ============================================================

print(model)

In [ ]:
# ============================================================
# PARAMETER MODEL
# ============================================================

model.get_params()

# Kesimpulan Tahap Pelatihan

Pada tahap ini proses pelatihan model telah berhasil dilakukan menggunakan algoritma Random Forest Classifier.

Model telah mempelajari pola hubungan antara parameter tanah dan lingkungan terhadap jenis tanaman yang sesuai.

Model yang telah dilatih selanjutnya akan dievaluasi menggunakan data testing untuk mengetahui tingkat akurasi dan kualitas prediksi.

# 14. Evaluasi Model

Setelah model Random Forest selesai dilatih, tahap berikutnya adalah melakukan evaluasi terhadap performa model menggunakan data testing.

Evaluasi dilakukan menggunakan beberapa metrik, yaitu:

- Accuracy
- Classification Report
- Confusion Matrix
- Feature Importance

Hasil evaluasi digunakan untuk mengetahui kemampuan model dalam memberikan prediksi terhadap data yang belum pernah dilihat sebelumnya.

In [ ]:
# ============================================================
# MELAKUKAN PREDIKSI
# ============================================================

y_pred = model.predict(X_test)

print("Prediksi berhasil dilakukan.")

# 15. Menghitung Akurasi Model

Akurasi merupakan persentase jumlah prediksi yang benar dibandingkan dengan seluruh data pengujian.

Semakin tinggi nilai akurasi, maka semakin baik kemampuan model dalam melakukan klasifikasi.

In [ ]:
# ============================================================
# AKURASI MODEL
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("="*50)
print("AKURASI MODEL")
print("="*50)

print(f"Akurasi Model : {accuracy*100:.2f}%")

# 16. Classification Report

Classification Report digunakan untuk mengetahui performa model pada setiap kelas tanaman.

Informasi yang ditampilkan meliputi:

- Precision
- Recall
- F1-Score
- Support

Metrik tersebut memberikan gambaran yang lebih lengkap dibandingkan hanya melihat nilai akurasi.

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print(classification_report(

    y_test,

    y_pred,

    target_names=label_encoder.classes_

))

# 17. Confusion Matrix

Confusion Matrix digunakan untuk melihat jumlah prediksi yang benar maupun yang salah pada setiap kelas tanaman.

Visualisasi ini mempermudah analisis terhadap hasil klasifikasi model.

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(

    y_test,

    y_pred

)

plt.figure(figsize=(12,10))

disp = ConfusionMatrixDisplay(

    confusion_matrix=cm,

    display_labels=label_encoder.classes_

)

disp.plot(

    xticks_rotation=90,

    cmap="Greens"

)

plt.title("Confusion Matrix")

plt.show()

# 18. Feature Importance

Random Forest mampu menunjukkan tingkat kepentingan masing-masing fitur terhadap hasil prediksi.

Semakin tinggi nilai Feature Importance, maka semakin besar pengaruh fitur tersebut dalam menentukan jenis tanaman yang direkomendasikan.

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({

    "Fitur": X.columns,

    "Importance": model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

importance

In [ ]:
# ============================================================
# VISUALISASI FEATURE IMPORTANCE
# ============================================================

plt.figure(figsize=(10,6))

plt.bar(

    importance["Fitur"],

    importance["Importance"],

    color="green"

)

plt.title("Feature Importance")

plt.xlabel("Fitur")

plt.ylabel("Importance")

plt.xticks(rotation=45)

plt.show()

# 19. Analisis Hasil Evaluasi

Berdasarkan hasil evaluasi yang telah dilakukan, diperoleh beberapa kesimpulan sebagai berikut:

1. Model Random Forest mampu melakukan klasifikasi jenis tanaman dengan tingkat akurasi yang tinggi.
2. Classification Report menunjukkan nilai Precision, Recall, dan F1-Score yang baik pada sebagian besar kelas tanaman.
3. Confusion Matrix memperlihatkan bahwa sebagian besar data berhasil diprediksi dengan benar.
4. Feature Importance menunjukkan bahwa seluruh parameter memiliki kontribusi terhadap proses prediksi, meskipun tingkat pengaruhnya berbeda-beda.

Secara keseluruhan, model Random Forest layak digunakan sebagai mesin rekomendasi pada aplikasi Smart Farming Assistant.

# 20. Menyimpan Model

Setelah model berhasil dilatih dan dievaluasi, tahap berikutnya adalah menyimpan model agar dapat digunakan kembali tanpa perlu melakukan proses pelatihan ulang.

Pada penelitian ini disimpan dua file utama, yaitu:

- **crop_model.pkl** → Model Random Forest yang telah dilatih.
- **label_encoder.pkl** → Digunakan untuk mengubah hasil prediksi numerik menjadi nama tanaman.

Kedua file tersebut akan digunakan pada aplikasi berbasis Streamlit sebagai mesin prediksi.

In [ ]:
# ============================================================
# MENYIMPAN MODEL
# ============================================================

joblib.dump(model, "crop_model.pkl")

print("Model berhasil disimpan sebagai crop_model.pkl")

In [ ]:
# ============================================================
# MENYIMPAN LABEL ENCODER
# ============================================================

joblib.dump(label_encoder, "label_encoder.pkl")

print("Label Encoder berhasil disimpan sebagai label_encoder.pkl")

# 21. Simulasi Prediksi

Tahap ini bertujuan untuk menguji model menggunakan data baru.

Nilai yang dimasukkan merupakan contoh kondisi tanah dan lingkungan yang kemudian diproses oleh model untuk menghasilkan rekomendasi tanaman.

In [ ]:
# ============================================================
# CONTOH DATA BARU
# ============================================================

sample = [[90, 42, 43, 20.8, 82.0, 6.5, 202.9]]

prediction = model.predict(sample)

hasil = label_encoder.inverse_transform(prediction)

print("="*50)
print("HASIL PREDIKSI")
print("="*50)

print("Tanaman yang direkomendasikan adalah :")

print(hasil[0])

# 22. Kesimpulan

Berdasarkan seluruh proses yang telah dilakukan, dapat disimpulkan bahwa:

1. Dataset Crop Recommendation berhasil diproses dengan baik tanpa ditemukan missing value.

2. Proses preprocessing berhasil dilakukan menggunakan Label Encoder sehingga data target dapat diproses oleh algoritma Random Forest.

3. Model Random Forest berhasil dilatih menggunakan data training dan mampu melakukan prediksi terhadap data testing dengan tingkat akurasi yang tinggi.

4. Model mampu memberikan rekomendasi tanaman berdasarkan parameter tanah dan lingkungan, yaitu Nitrogen (N), Phosphorus (P), Potassium (K), Temperature, Humidity, pH, dan Rainfall.

5. Model yang telah dibuat berhasil disimpan dalam bentuk file **crop_model.pkl** dan **label_encoder.pkl** sehingga dapat digunakan kembali pada aplikasi Smart Farming Assistant berbasis Streamlit.

Secara keseluruhan, model Random Forest layak digunakan sebagai mesin rekomendasi tanaman pada aplikasi Smart Farming Assistant.

# 23. Referensi

1. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research.

2. Breiman, L. (2001). Random Forests. Machine Learning.

3. McKinney, W. (2022). Python for Data Analysis (3rd Edition). O'Reilly Media.

4. Chollet, F. (2021). Deep Learning with Python (2nd Edition). Manning Publications.

5. Crop Recommendation Dataset. Kaggle.

# Terima Kasih

Notebook ini disusun sebagai dokumentasi proses pembangunan model Machine Learning pada proyek **Smart Farming Assistant**.

Model yang dihasilkan selanjutnya diintegrasikan ke dalam aplikasi berbasis **Streamlit** sehingga pengguna dapat memperoleh rekomendasi tanaman secara cepat berdasarkan kondisi tanah dan lingkungan.

**Nama:** Junaidy Muhtarom

**Program Studi:** Teknik Informatika

**Universitas:** Universitas PGRI Madiun